# PSO-ANN Revalidation, NHANES 2013-2014

**Author:** Ziang Zhang

The replicated paper reported 0.97 accuracy for an artificial neural network whose weights were initialized by particle swarm optimization (PSO-ANN). In the course project (v1) the PSO-ANN reached 0.81 accuracy, but it was evaluated on the leaky row-level split (about 75% of test rows had a copy in the SMOTE training set) and only by accuracy, which is not comparable to AUC at 4% prevalence.

This notebook re-evaluates the PSO-ANN in the corrected pipeline, with exactly what the logistic regression and random forest got in `nhanes_chd_audit_and_revalidation.ipynb`:

- the same 5,751 participants (one row each) and the same 19 predictors;
- the same participant-level stratified 5-fold split (`StratifiedKFold(5, shuffle=True, random_state=42)`);
- the same preprocessing (mean/mode imputation, z-scoring, one-hot encoding), fit inside each training fold;
- the same bootstrap (1,000 resamples, seed 42), so the comparison with logistic regression is paired.

**Model (paper and v1 settings):** one hidden layer with 5 neurons; PSO with 30 particles and 100 iterations finds the initial weights; backpropagation then trains them. No settings were tuned to improve the score. Choices the paper does not pin down are listed in section 2.

**Outputs:** `outputs/pso_ann_cv.csv`, `outputs/pso_ann_seeds.csv`.

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from scipy.special import expit
from sklearn.model_selection import train_test_split

# Works when run from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
SEED = 42
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 1. Cohort, folds and the logistic-regression reference

The logistic-regression out-of-fold predictions are regenerated with the audit notebook's code and checked against `outputs/auc_bootstrap_ci.csv` before anything else is computed.

In [2]:
# Same cohort, predictors, preprocessing and folds as the audit notebook (sections 2 and 5)
raw = pd.read_csv(DATA_DIR / "selected_features_v1.csv").rename(columns={"Blood-Rel Stroke": "StrokeHistory"})
labeled = raw[raw["disease_status"].notna()]
d = labeled.drop_duplicates("SEQN").reset_index(drop=True)
assert d["SEQN"].is_unique

categorical = ["Gender", "Vigorous Work", "Moderate Work", "Diabetes", "Blood-Rel Diabetes", "StrokeHistory"]
numeric = ["Age", "Systolic", "Diastolic", "Platelet Count", "Red-Cell Distribution Width", "Cholesterol",
           "Creatinine", "Glucose", "LDH", "Uric Acid", "Triglycerides", "HDL", "Glycohemoglobin"]
X = d[categorical + numeric].copy()
X[categorical] = X[categorical].astype(str)
y = (d["disease_status"] == "diseased").astype(int).to_numpy()

def make_preprocess():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="mean")), ("scale", StandardScaler())]), numeric),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(X, y))
test_idx_all = np.concatenate([te for _, te in folds])
assert len(test_idx_all) == len(y) and len(np.unique(test_idx_all)) == len(y), "each participant must be tested exactly once"
assert all(len(np.intersect1d(tr, te)) == 0 for tr, te in folds)
print(f"{len(y):,} adults, {y.sum()} CHD cases ({y.mean():.2%}), {len(categorical) + len(numeric)} predictors")

5,751 adults, 232 CHD cases (4.03%), 19 predictors


In [3]:
def bootstrap_auc(y_true, prob, n_boot=1000, seed=SEED):
    """Same function and seed as the audit notebook, so resamples are paired across models."""
    r = np.random.default_rng(seed)
    n = len(y_true)
    aucs = []
    for _ in range(n_boot):
        idx = r.integers(0, n, n)
        if y_true[idx].sum() == 0:
            continue
        aucs.append(roc_auc_score(y_true[idx], prob[idx]))
    aucs = np.array(aucs)
    return aucs.mean(), np.percentile(aucs, 2.5), np.percentile(aucs, 97.5), aucs

# Class-weighted logistic regression, out-of-fold (audit notebook section 5)
lr_oof = np.zeros(len(y))
for tr, te in folds:
    pipe = Pipeline([("prep", make_preprocess()), ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))])
    pipe.fit(X.iloc[tr], y[tr])
    lr_oof[te] = pipe.predict_proba(X.iloc[te])[:, 1]

_, lr_lo, lr_hi, lr_boot = bootstrap_auc(y, lr_oof)
lr_auc = roc_auc_score(y, lr_oof)
ref = pd.read_csv(OUT_DIR / "auc_bootstrap_ci.csv").set_index("Model").loc["Logistic regression"]
assert (round(lr_auc, 3), round(lr_lo, 3), round(lr_hi, 3)) == (ref["AUC"], ref["CI_low"], ref["CI_high"]), \
    "LR out-of-fold predictions do not reproduce outputs/auc_bootstrap_ci.csv"
print(f"Logistic regression (class-weighted): AUC {lr_auc:.3f} ({lr_lo:.3f}-{lr_hi:.3f}), matches outputs/auc_bootstrap_ci.csv")

Logistic regression (class-weighted): AUC 0.859 (0.837-0.881), matches outputs/auc_bootstrap_ci.csv


## 2. PSO-ANN

Fixed by the paper and v1: 5 hidden neurons, PSO with 30 particles and 100 iterations for the initial weights, then backpropagation.

Choices the paper leaves open, set to textbook defaults and not tuned:

| Choice | Setting | Reason |
|---|---|---|
| Activations | sigmoid hidden and output layer | classic ANN of the paper's era; output is a probability |
| PSO objective | cross-entropy on the training part of the fold | the same loss backpropagation minimizes |
| PSO coefficients | inertia 0.729, c1 = c2 = 1.494, positions U(-1, 1), velocity clipped to ±1 | Clerc-Kennedy constriction values, the standard default |
| Backpropagation | full-batch Adam, learning rate 0.01 | default Adam step size; full batch because the data are small |
| Stopping | 20% of each training fold (stratified) held out; stop after 50 epochs without improvement in validation loss, at most 2,000 epochs; keep the best weights | the network must not see the test fold, so early stopping needs its own validation data |
| Class imbalance | balanced class weights in the loss (main); unweighted (for calibration); SMOTE on the training part (as in v1) | weighted matches the corrected LR and RF; unweighted matches the calibrated LR; SMOTE reproduces v1's approach correctly, inside the fold |

The network sees 80% of each training fold because 20% is reserved for early stopping; logistic regression uses all of it. All preprocessing, PSO and SMOTE are fit on the training part only. The gradient is checked numerically below.

In [4]:
H = 5  # hidden neurons

def unpack(theta, k):
    W1 = theta[:k * H].reshape(k, H)
    b1 = theta[k * H:k * H + H]
    W2 = theta[k * H + H:k * H + 2 * H]
    return W1, b1, W2, theta[-1]

def forward(theta, A, k):
    W1, b1, W2, b2 = unpack(theta, k)
    h = expit(A @ W1 + b1)
    return expit(h @ W2 + b2), h

def loss(theta, A, t, w, k):
    p = np.clip(forward(theta, A, k)[0], 1e-7, 1 - 1e-7)
    return -(w * (t * np.log(p) + (1 - t) * np.log(1 - p))).sum() / w.sum()

def grad(theta, A, t, w, k):
    W1, b1, W2, b2 = unpack(theta, k)
    p, h = forward(theta, A, k)
    dz2 = w * (p - t) / w.sum()
    dh = np.outer(dz2, W2) * h * (1 - h)
    return np.concatenate([(A.T @ dh).ravel(), dh.sum(0), h.T @ dz2, [dz2.sum()]])

def pso(A, t, w, k, rng, n_particles=30, n_iter=100, inertia=0.729, c1=1.494, c2=1.494):
    """Global-best PSO over all network weights; returns the best particle."""
    P = k * H + 2 * H + 1
    pos = rng.uniform(-1, 1, (n_particles, P))
    vel = rng.uniform(-0.1, 0.1, (n_particles, P))
    pbest, pval = pos.copy(), np.array([loss(q, A, t, w, k) for q in pos])
    g = pbest[pval.argmin()].copy()
    for _ in range(n_iter):
        r1, r2 = rng.random((n_particles, P)), rng.random((n_particles, P))
        vel = np.clip(inertia * vel + c1 * r1 * (pbest - pos) + c2 * r2 * (g - pos), -1, 1)
        pos = pos + vel
        val = np.array([loss(q, A, t, w, k) for q in pos])
        better = val < pval
        pbest[better], pval[better] = pos[better], val[better]
        g = pbest[pval.argmin()].copy()
    return g

def backprop(theta, A, t, w, Av, tv, wv, k, lr=0.01, max_epochs=2000, patience=50):
    """Full-batch Adam with early stopping on the validation loss; returns best weights, epochs run, best epoch."""
    m, v = np.zeros_like(theta), np.zeros_like(theta)
    best, best_val, best_epoch, wait = theta.copy(), loss(theta, Av, tv, wv, k), 0, 0
    for epoch in range(1, max_epochs + 1):
        g = grad(theta, A, t, w, k)
        m, v = 0.9 * m + 0.1 * g, 0.999 * v + 0.001 * g * g
        theta = theta - lr * (m / (1 - 0.9 ** epoch)) / (np.sqrt(v / (1 - 0.999 ** epoch)) + 1e-8)
        val = loss(theta, Av, tv, wv, k)
        if val < best_val - 1e-6:
            best, best_val, best_epoch, wait = theta.copy(), val, epoch, 0
        else:
            wait += 1
            if wait >= patience:
                break
    return best, epoch, best_epoch

def balanced_weights(t):
    n, n1 = len(t), t.sum()
    return np.where(t == 1, n / (2 * n1), n / (2 * (n - n1)))

# Numerical gradient check on random weights
_A = np.random.default_rng(0).normal(size=(200, 8)); _t = (np.random.default_rng(1).random(200) < 0.2).astype(float)
_w = balanced_weights(_t); _th = np.random.default_rng(2).normal(0, 0.5, 8 * H + 2 * H + 1)
_num = np.array([(loss(_th + e, _A, _t, _w, 8) - loss(_th - e, _A, _t, _w, 8)) / 2e-6 for e in np.eye(len(_th)) * 1e-6])
_err = np.max(np.abs(grad(_th, _A, _t, _w, 8) - _num)) / np.max(np.abs(_num))
assert _err < 1e-6, _err
print(f"Gradient check: max relative error {_err:.1e}")

Gradient check: max relative error 7.6e-10


In [5]:
def pso_ann_oof(variant, seed=SEED):
    """Out-of-fold PSO-ANN probabilities. variant: 'weighted', 'unweighted' or 'smote'."""
    prob, log = np.zeros(len(y)), []
    for f, (tr, te) in enumerate(folds):
        s = seed * 100 + f
        rng = np.random.default_rng(s)
        fit_idx, val_idx = train_test_split(tr, test_size=0.2, stratify=y[tr], random_state=s)
        prep = make_preprocess().fit(X.iloc[fit_idx])
        A, Av, At = (np.asarray(prep.transform(X.iloc[i]), dtype=float) for i in (fit_idx, val_idx, te))
        t, tv = y[fit_idx].astype(float), y[val_idx].astype(float)
        if variant == "smote":
            A, t = SMOTE(random_state=SEED).fit_resample(A, t)
        if variant == "weighted":
            w, wv = balanced_weights(t), balanced_weights(tv)
        else:
            w, wv = np.ones(len(t)), np.ones(len(tv))
        k = A.shape[1]
        theta0 = pso(A, t, w, k, rng)
        theta, epochs, best_epoch = backprop(theta0, A, t, w, Av, tv, wv, k)
        prob[te] = forward(theta, At, k)[0]
        log.append({"Variant": variant, "Fold": f + 1, "Inputs": k, "Weights": len(theta),
                    "PSO train loss": loss(theta0, A, t, w, k), "PSO val loss": loss(theta0, Av, tv, wv, k),
                    "Final val loss": loss(theta, Av, tv, wv, k), "Epochs run": epochs, "Best epoch": best_epoch})
    return prob, log

VARIANTS = {"weighted": "PSO-ANN (class-weighted)", "unweighted": "PSO-ANN (unweighted)",
            "smote": "PSO-ANN (SMOTE inside each fold)"}
ann_oof, logs = {}, []
for v in VARIANTS:
    ann_oof[v], lg = pso_ann_oof(v)
    logs += lg
fold_log = pd.DataFrame(logs).round(4)
fold_log

,Variant,Fold,Inputs,Weights,PSO train loss,PSO val loss,Final val loss,Epochs run,Best epoch
0,weighted,1,29,156,0.3918,0.5450,0.5450,50,0
1,weighted,2,31,166,0.4259,0.4691,0.4691,50,0
2,weighted,3,31,166,0.3965,0.4384,0.4250,77,27
3,weighted,4,31,166,0.4007,0.5281,0.5268,51,1
4,weighted,5,29,156,0.3827,0.6213,0.6213,50,0
5,unweighted,1,29,156,0.1194,0.1456,0.1456,50,0
6,unweighted,2,31,166,0.1290,0.1308,0.1308,51,1
7,unweighted,3,31,166,0.1214,0.1281,0.1280,56,6
8,unweighted,4,31,166,0.1208,0.1400,0.1399,51,1
9,unweighted,5,29,156,0.1199,0.1518,0.1498,146,96


**Reading the fold log.** After encoding, the network has 29-31 inputs (a category can be absent from a training fold) and 156-166 weights, trained on about 150 cases. For the class-weighted and unweighted networks, `Best epoch` is 0-1 in most folds: the PSO weights already fit the training part better than the validation part, and from the first epochs backpropagation lowered the training loss while raising the validation loss, so early stopping kept (nearly) the PSO weights. With SMOTE, backpropagation trained for 129-422 epochs before the validation loss stopped improving. This is reported as is; the stopping rule was fixed before the results were seen.

## 3. Performance

AUC with the same paired bootstrap as LR vs RF in the audit notebook; calibration slope and Brier score as in its section 10; sensitivity and specificity at the 0.40 screening threshold for the models that are trained to balance the classes.

In [6]:
def calibration_slope(y_true, prob):
    p = np.clip(prob, 1e-6, 1 - 1e-6)
    lp = np.log(p / (1 - p)).reshape(-1, 1)
    return LogisticRegression(penalty=None, max_iter=1000).fit(lp, y_true).coef_[0, 0]

def sens_spec(y_true, prob, t=0.40):
    pred = prob >= t
    return (pred & (y_true == 1)).sum() / (y_true == 1).sum(), (~pred & (y_true == 0)).sum() / (y_true == 0).sum()

lr_unweighted = np.zeros(len(y))
for tr, te in folds:
    pipe = Pipeline([("prep", make_preprocess()), ("model", LogisticRegression(max_iter=2000))]).fit(X.iloc[tr], y[tr])
    lr_unweighted[te] = pipe.predict_proba(X.iloc[te])[:, 1]

rows = []
for name, prob, balanced in [("Logistic regression (class-weighted)", lr_oof, True),
                             ("Logistic regression (unweighted)", lr_unweighted, False)] + \
                            [(VARIANTS[v], ann_oof[v], v != "unweighted") for v in VARIANTS]:
    _, lo, hi, b = bootstrap_auc(y, prob)
    diff = lr_boot - b
    se, sp = sens_spec(y, prob) if balanced else (np.nan, np.nan)
    rows.append({"Model": name, "AUC": roc_auc_score(y, prob), "CI_low": lo, "CI_high": hi,
                 "LR_minus_model": diff.mean(), "Diff_CI_low": np.percentile(diff, 2.5), "Diff_CI_high": np.percentile(diff, 97.5),
                 "CalibrationSlope": calibration_slope(y, prob), "Brier": brier_score_loss(y, prob),
                 "MeanPredicted": prob.mean(), "Sensitivity@0.40": se, "Specificity@0.40": sp})
perf = pd.DataFrame(rows)
perf.loc[perf["Model"] == "Logistic regression (class-weighted)", ["LR_minus_model", "Diff_CI_low", "Diff_CI_high"]] = np.nan
perf = perf.round(4)

# Leakage guard: in this cohort, honest models sit near 0.86 (LR 0.859, RF 0.858)
assert perf["AUC"].max() < 0.90, "AUC above 0.90: check for leakage before reporting"
perf.to_csv(OUT_DIR / "pso_ann_cv.csv", index=False)
print(f"Brier of the no-skill model: {brier_score_loss(y, np.full(len(y), y.mean())):.4f}")
perf

Brier of the no-skill model: 0.0387


,Model,AUC,CI_low,CI_high,LR_minus_model,Diff_CI_low,Diff_CI_high,CalibrationSlope,Brier,MeanPredicted,Sensitivity@0.40,Specificity@0.40
0,Logistic regression (class-weighted),0.8589,0.8367,0.8811,NaN,NaN,NaN,0.7750,0.1506,0.3008,0.8621,0.7168
1,Logistic regression (unweighted),0.8604,0.8376,0.8822,-0.0014,-0.0052,0.0023,0.9380,0.0352,0.0405,NaN,NaN
2,PSO-ANN (class-weighted),0.8361,0.8115,0.8601,0.0230,0.0109,0.0350,0.6975,0.1501,0.2681,0.8190,0.7369
3,PSO-ANN (unweighted),0.8362,0.8109,0.8602,0.0229,0.0110,0.0358,0.7430,0.0366,0.0412,NaN,NaN
4,PSO-ANN (SMOTE inside each fold),0.8032,0.7775,0.8292,0.0558,0.0392,0.0718,0.4055,0.1276,0.2059,0.6509,0.8036


## 4. Stability across random seeds

The network, PSO and the early-stopping split depend on the random seed. The whole cross-validation is repeated with five seeds (the folds stay fixed) to show the spread. Seed 42 remains the reported result; this is not a search for the best seed.

In [7]:
SEEDS = [42, 43, 44, 45, 46]
seed_rows = []
for v in VARIANTS:
    for s in SEEDS:
        p = ann_oof[v] if s == SEED else pso_ann_oof(v, seed=s)[0]
        seed_rows.append({"Model": VARIANTS[v], "Seed": s, "AUC": roc_auc_score(y, p),
                          "CalibrationSlope": calibration_slope(y, p), "Brier": brier_score_loss(y, p)})
seeds = pd.DataFrame(seed_rows).round(4)
seeds.to_csv(OUT_DIR / "pso_ann_seeds.csv", index=False)
seeds.groupby("Model")[["AUC", "CalibrationSlope", "Brier"]].agg(["mean", "min", "max"]).round(3)

AUC               CalibrationSlope         \
                                   mean    min    max             mean    min   
Model                                                                           
PSO-ANN (SMOTE inside each fold)  0.812  0.790  0.828            0.455  0.399   
PSO-ANN (class-weighted)          0.840  0.834  0.851            0.711  0.698   
PSO-ANN (unweighted)              0.842  0.836  0.844            0.772  0.743   

                                         Brier                
                                    max   mean    min    max  
Model                                                         
PSO-ANN (SMOTE inside each fold)  0.521  0.130  0.127  0.134  
PSO-ANN (class-weighted)          0.720  0.151  0.147  0.154  
PSO-ANN (unweighted)              0.799  0.036  0.036  0.037

## 5. Conclusions

- **The PSO-ANN does not beat logistic regression once the leakage is removed.** Out-of-fold AUC 0.836 (95% CI 0.812-0.860) for the class-weighted network, 0.023 below the class-weighted logistic regression (paired bootstrap 95% CI 0.011 to 0.035). Across five seeds its AUC was 0.834-0.851, every one below 0.859.
- **Calibration:** the unweighted network gets the average risk right (mean predicted 4.12% vs observed 4.03%) but its calibration slope is 0.74 (predictions too extreme); Brier 0.0366 vs 0.0352 for the unweighted logistic regression (no-skill 0.0387). The class-weighted network inflates risk, as class weighting does for logistic regression (mean predicted 26.8%, slope 0.70).
- **SMOTE, the v1 approach, applied correctly inside each fold, does worse:** AUC 0.803 (0.778-0.829), calibration slope 0.41.
- **What this does and does not show:** Under the paper's specification and without tuning, PSO-ANN (AUC 0.836) did not match logistic regression (0.859); backpropagation added little after PSO initialization and the network trained on 80% of each fold, so this does not show that neural networks cannot help with these predictors.
- **The paper's 0.97 and v1's 0.81 are accuracies**, which at 4% prevalence are dominated by the majority class (predicting "no CHD" for everyone scores 0.96), and v1's was measured on a leaky split. Neither is comparable to the AUC reported here.
